# inference.ipynb

Notebook ini dijalankan pada benchmark riil 2021-2023 dan harus dibaca bersama artefak provenance, comparison report, serta manual review summary.

Benchmark saat ini memakai slice data riil tahun 2021-2023; lihat `data/processed/source_manifest.json`, `models/benchmark_comparison.json`, dan `data/processed/manual_review_summary.csv` untuk konteks synthetic-vs-real dan dampak manual review.

Jika row-level reviewed labels tersedia nanti, impor dengan `scripts/import_reviewed_row_level.py` lalu rerun `scripts/run_diagnostics.py` agar artefak reviewed benchmark ikut terbarui.


In [1]:
import warnings
warnings.filterwarnings("ignore", category=Warning, module="tqdm")
warnings.filterwarnings("ignore", message="IProgress not found")

from pathlib import Path
import pandas as pd
from src.explain import explain_single, get_explainer, shap_counterfactual
from src.narrative import render_explanation_narrative

assert Path("models/xgb_model.ubj").exists(), "Jalankan training.ipynb terlebih dahulu atau siapkan model lokal"
X_test = pd.read_parquet("test_data/features.parquet")
row = X_test.iloc[[0]]
row.head()


,f_tender_value_log,f_award_value_log,f_price_deviation_ratio,f_main_procurement_category_enc,f_award_duration_days,f_tender_items_count,f_award_items_count,f_title_length,f_description_length,f_tender_value_missing,...,f_supplier_recent_90d_award_count,f_buyer_value_growth_rate,f_supplier_capacity_ratio,f_buyer_hist_avg_award,f_buyer_hist_award_std,f_supplier_hist_avg_award,f_buyer_unique_suppliers_count,f_supplier_unique_buyers_count,f_pair_share_of_buyer_history,f_pair_share_of_supplier_history
0,NaN,NaN,NaN,-1.0,NaN,0,0.0,55.0,55.0,1.0,...,0.0,NaN,NaN,9.220749e+08,1.877380e+09,NaN,63.0,0.0,0.0,0.0


In [2]:
import xgboost as xgb
model = xgb.Booster()
model.load_model("models/xgb_model.ubj")
explainer = get_explainer(model)
explanation = explain_single(row, model=model, explainer=explainer)
explanation


{'predicted_class': 1,
 'predicted_label': 'Medium Risk',
 'probability': 0.999981,
 'probabilities': [1.9e-05, 0.999981, 0.0],
 'factors': [{'feature': 'f_title_length',
   'value': 55.0,
   'feature_value': 55.0,
   'shap_value': 1.604941964149475,
   'direction': 'increases_risk'},
  {'feature': 'f_buyer_supplier_repeat_count',
   'value': 0.0,
   'feature_value': 0.0,
   'shap_value': 0.6987288594245911,
   'direction': 'increases_risk'},
  {'feature': 'f_supplier_recent_90d_award_count',
   'value': 0.0,
   'feature_value': 0.0,
   'shap_value': 0.5464428663253784,
   'direction': 'increases_risk'},
  {'feature': 'f_is_q4',
   'value': 0.0,
   'feature_value': 0.0,
   'shap_value': 0.5251885652542114,
   'direction': 'increases_risk'},
  {'feature': 'f_price_deviation_ratio',
   'value': None,
   'feature_value': None,
   'shap_value': 0.3777310848236084,
   'direction': 'increases_risk'}]}

In [3]:
counterfactual = shap_counterfactual(explanation, target_class=0)
render_explanation_narrative(explanation, counterfactual)


'Peringkat investigatif paket ini adalah **Perlu Pantauan**.\nDiturunkan dari triase model: Medium Risk dengan probabilitas 100.00%.\nModel mengklasifikasikan paket ini sebagai **Medium Risk** dengan probabilitas 100.00%.\nCatatan penting: ini adalah triase risiko, bukan bukti fraud final. Status kritis hanya boleh dinaikkan bila ada bukti resmi yang terhubung.\nFaktor yang paling memengaruhi prediksi adalah:\n- Panjang judul tender bernilai 55 dan meningkatkan skor risiko dengan kontribusi SHAP sekitar 1.6049.\n- Frekuensi hubungan buyer-supplier berulang bernilai 0 dan meningkatkan skor risiko dengan kontribusi SHAP sekitar 0.6987.\n- F supplier recent 90d award count bernilai 0 dan meningkatkan skor risiko dengan kontribusi SHAP sekitar 0.5464.\n- Waktu publikasi pada kuartal iv bernilai 0 dan meningkatkan skor risiko dengan kontribusi SHAP sekitar 0.5252.\n- Rasio deviasi harga terhadap nilai tender bernilai tidak tersedia dan meningkatkan skor risiko dengan kontribusi SHAP sekitar

In [4]:
import numpy as np
import onnxruntime as rt

if Path("models/xgb_model.onnx").exists():
    sess = rt.InferenceSession("models/xgb_model.onnx")
    input_name = sess.get_inputs()[0].name
    sample = row.to_numpy(dtype=np.float32)
    outputs = sess.run(None, {input_name: sample})
    print({
        "onnx_output_count": len(outputs),
        "onnx_output_shapes": [getattr(o, 'shape', None) for o in outputs],
        "first_output_preview": outputs[0][0].tolist() if len(outputs) > 0 else None,
    })
else:
    print("models/xgb_model.onnx belum tersedia")


{'onnx_output_count': 2, 'onnx_output_shapes': [(1,), (1, 3)], 'first_output_preview': 1}
